In [ ]:
import sys, os, json
import numpy as np, pandas as pd
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)
from Analysis.data import builders, loaders, tools, viz

ROOT_LOG = "/home/nonato/GitProjects/Airob/log"  

def print_dict(dictionary:dict):
    for key, content in dictionary.items():
        print(f"{key}: {content}")

# Do this once to create an archive of all experiments in parquet
# loaders.put_data_together(ROOT_LOG)

#Do this to load the parquet file created.
# df, fitNames, minmaxValues = loaders.load_parquet_log(os.path.join(ROOT_LOG, "completeData.parquet"))
# with open(os.path.join(ROOT_LOG, "taskMaps.json"), "r") as f:
#     taskMaps = json.load(f)
# print(df["experiment"].unique())
# print(df.shape) #names are wrong...grid is 20x5
# df["shape"].unique

: 

In [ ]:
df, fitNames, minmaxValues = loaders.load_parquet_log("/home/non4to/Gut Hib/Airob/log/completeData.parquet")

fitnessData = builders.build_fitness_data(df)
outputFolder = "/home/non4to/Gut Hib/Airob/log"

viz._print_line_graph(
    data=fitnessData,
    outputPath= outputFolder,
    title="Evolution of mean fitness",
    xLabel="Generation",
    yLabel="Mean Fitness"

In [ ]:
# Get top [%] bots from experiment folder 
output = viz.get_top_bots_percentage(logdir="/home/non4to/Gut Hib/Airob/log/mixed-randomSelectAge50-20x5_seed7_CGA_08271513", topPerc=0.05)
print_dict(output)

In [ ]:
# CONTAR ROBOS UNICOS DE TODOS OS DADOS

import numpy as np
import pandas as pd

df["shape_code"] = df["shape"].apply(lambda s: "".join(map(str, s.flatten())))
df["shape_code"].head()

# 2. Filtra o DataFrame mantendo apenas a PRIMEIRA ocorrência de cada formato único
df_unique_shapes = df.drop_duplicates(subset=["shape_code"]).copy()
df_unique_shapes = df_unique_shapes.reset_index(drop=True)

# 3. Exibe o diagnóstico da redução do dataset
total_populacao = len(df)
total_unicos = len(df_unique_shapes)
taxa_clones = (1 - (total_unicos / total_populacao)) * 100

print("=== DIAGNÓSTICO DE FILTRAGEM DE SHAPES ÚNICOS ===")
print(f"Total de robôs na população histórica: {total_populacao}")
print(f"Total de formatos (shapes) únicos isolados: {total_unicos}")
print(f"Taxa de redundância (clones/repetições): {taxa_clones:.2f}%")

# CHECAGEM 1: Qual o tamanho do código gerado? (Deve ser exatamente 25 para uma matriz 5x5)
tamanhos_codigos = df["shape_code"].str.len().unique()
print(f"Tamanhos de string encontrados: {tamanhos_codigos}")  # Deve imprimir [25]

# CHECAGEM 2: Quais são os formatos mais populosos do seu experimento?
print("\nTop 5 formatos (shapes) mais repetidos de toda a história:")
print(df["shape_code"].value_counts().head(5))

In [ ]:
# PROCURAR valor ideal de ers pro dbscan

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# =====================================================================
# 1. PREPARAÇÃO DOS DADOS (APENAS SHAPES ÚNICOS)
# =====================================================================
# Pega as matrizes 5x5 do df_unique_shapes e empilha em um vetor (5803, 25)
X_unique = np.stack(
    df_unique_shapes["shape"].apply(lambda s: s.flatten()).values
)

# =====================================================================
# 2. CÁLCULO DA DISTÂNCIA AO k-ÉSIMO VIZINHO
# =====================================================================
k = 4  # Define k igual ao min_cluster_size pretendido

# Instancia o algoritmo para buscar os k vizinhos mais próximos usando Hamming
vizinhos = NearestNeighbors(
    n_neighbors=k, metric="hamming", algorithm="brute"
)
vizinhos.fit(X_unique)

# Calcula as distâncias para cada um dos 5.803 shapes únicos
distancias, _ = vizinhos.kneighbors(X_unique)

# Pega a distância exatamente para o k-ésimo vizinho (última coluna)
k_distancias = distancias[:, -1]

# Ordena do MAIOR para o MENOR (ordem decrescente, como no paper original)
k_distancias_ordenadas = np.sort(k_distancias)[::-1]

# =====================================================================
# 3. PLOTAGEM DO GRÁFICO (PAPER ESTER ET AL., 1996)
# =====================================================================
plt.figure(figsize=(10, 6), dpi=120)

# Plota a curva dos 5.803 shapes únicos
plt.plot(
    k_distancias_ordenadas, color="#1f77b4", linewidth=2.5, label=f"Curva {k}-dist"
)

# Configurações de eixos e marcas discretas de Hamming (passos de 0.04)
plt.title(
    f"Gráfico de k-Distância no Morfoespaço Único (N = {len(X_unique)} Shapes)",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Formatos (Shapes) Ordenados por Distância", fontsize=12)
plt.ylabel(f"Distância de Hamming ao {k}º Vizinho", fontsize=12)

# Coloca marcas no eixo Y em passos de 0.04 (1 célula de diferença no 5x5)
plt.yticks(np.arange(0, np.max(k_distancias_ordenadas) + 0.08, 0.04))
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import DBSCAN, HDBSCAN

def build_genotype_clusters(df: pd.DataFrame, eps: float = 0.2, min_samples: int = 5):
    clean_df = df.drop_duplicates(subset=["experiment", "seed", "id"]).copy().reset_index(drop=True)
    X = np.stack(clean_df["shape"].apply(lambda s: s.flatten()).values)
    clustering = DBSCAN(eps=eps, min_samples=min_samples, metric="hamming", algorithm="ball_tree")
    clean_df["cluster"] = clustering.fit(X).labels_   # -1 = ruído (sem grupo)
    
    return clean_df

def check_cluester_out(c):
    counts = c["cluster"].value_counts()
    print("Porcentagem de ruído:", (c["cluster"] == -1).mean() * 100, "%")
    print("Tamanho médio dos clusters:", counts[counts.index != -1].mean())
    print("Clusters com exatamente 10 amostras:", (counts == 10).sum())
    print()
    
def diagnose_clusters(c):
    counts = c["cluster"].value_counts()
    noise_count = counts.get(-1, 0)
    clusters_only = counts[counts.index != -1]
    
    print("=== DIAGNÓSTICO DE DISTRIBUIÇÃO ===")
    print(f"Total de Matrizes: {len(c)}")
    print(f"Total de Clusters válidos: {len(clusters_only)}")
    print(f"Ruído (-1): {noise_count} matrizes ({(noise_count/len(c))*100:.2f}%)")
    print("-" * 35)
    print(f"Maior cluster único: {clusters_only.max()} matrizes ({(clusters_only.max()/len(c))*100:.1f}% do total)")
    print(f"Mediana do tamanho dos clusters: {clusters_only.median()} matrizes")
    print(f"Menor cluster: {clusters_only.min()} matrizes")
    print("-" * 35)
    print("Tamanho dos 10 maiores clusters:")
    print(clusters_only.head(10).to_string())

import numpy as np
from scipy.stats import mode

def inspect_top_clusters(seed_df: pd.DataFrame, top_n: int = 5):
    counts = seed_df["cluster"].value_counts()
    top_clusters = counts[counts.index != -1].head(top_n).index
    
    print(f"=== INSPEÇÃO DOS {top_n} MAIORES CLUSTERS (Total de Robôs: {len(seed_df)}) ===\n")
    for rank, c_id in enumerate(top_clusters, 1):
        # Filtra as matrizes pertencentes a este cluster
        matrices = np.stack(seed_df[seed_df["cluster"] == c_id]["shape"].values)
        
        # Calcula o valor categórico mais frequente (moda) em cada célula 5x5
        modal_matrix = mode(matrices, axis=0).mode
        
        total_robots = counts[c_id]
        pct = (total_robots / len(seed_df)) * 100
        
        print(f"RANK {rank} | Cluster ID: {c_id} | {total_robots} robôs ({pct:.1f}%)")
        print(modal_matrix)
        print("-" * 45)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors


def plot_k_distance_graph_full(
    df: pd.DataFrame, k: int = 10, eps_highlight: float = 0.16
):
    # 1. Limpeza: apaga IDs iguais dentro da mesma seed e do mesmo experimento
    clean_df = df.drop_duplicates(
        subset=["experiment", "seed", "id"]
    ).reset_index(drop=True)

    # 2. Empilha as matrizes 5x5 em vetores unidimensionais de tamanho 25
    X = np.stack(clean_df["shape"].apply(lambda s: s.flatten()).values)

    print(f"Total de robôs únicos no dataset inteiro: {len(X)}")

    # 3. Calcula a distância ao k-ésimo vizinho usando Hamming
    nbrs = NearestNeighbors(
        n_neighbors=k, metric="hamming", algorithm="brute"
    ).fit(X)
    distances, _ = nbrs.kneighbors(X)

    # Distância ao k-ésimo vizinho (última coluna)
    k_distances = distances[:, -1]
    k_distances_sorted = np.sort(k_distances)[::-1]  # Ordem decrescente

    # 4. Plotagem com 2 visões: Visão Geral e Zoom na Curva
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), dpi=120)

    # --- GRÁFICO 1: VISÃO GERAL COMPLETA ---
    ax1.plot(
        k_distances_sorted,
        color="#1f77b4",
        linewidth=2,
        label=f"Curva {k}-dist",
    )
    if eps_highlight:
        ax1.axhline(
            y=eps_highlight,
            color="red",
            linestyle="--",
            linewidth=1.5,
            label=f"eps calibrado = {eps_highlight}",
        )
    ax1.set_title(
        f"1. Visão Geral (Dataset Inteiro: {len(X)} robôs)", fontweight="bold"
    )
    ax1.set_xlabel("Robôs Ordenados")
    ax1.set_ylabel(f"Distância de Hamming ({k}-dist)")
    ax1.set_yticks(np.arange(0, 0.44, 0.04))
    ax1.grid(True, linestyle=":", alpha=0.6)
    ax1.legend()

    # --- GRÁFICO 2: ZOOM NA ZONA DE TRANSIÇÃO (ONDE VARIAR) ---
    # Foca nas primeiras posições onde a distância é maior que zero
    non_zero_count = np.sum(k_distances_sorted > 0)
    zoom_limit = max(int(non_zero_count * 1.5), 1000)

    ax2.plot(
        k_distances_sorted[:zoom_limit],
        color="#1f77b4",
        linewidth=2,
        label=f"Curva {k}-dist",
    )
    if eps_highlight:
        ax2.axhline(
            y=eps_highlight,
            color="red",
            linestyle="--",
            linewidth=1.5,
            label=f"eps calibrado = {eps_highlight}",
        )
    ax2.set_title(
        f"2. Zoom na Transição (Primeiros {zoom_limit} robôs)",
        fontweight="bold",
    )
    ax2.set_xlabel("Robôs Ordenados (Apenas região não-nula)")
    ax2.set_ylabel(f"Distância de Hamming ({k}-dist)")
    ax2.set_yticks(np.arange(0, 0.44, 0.04))
    ax2.grid(True, linestyle=":", alpha=0.6)
    ax2.legend()

    plt.tight_layout()
    plt.show()


# Executa para o DataFrame completo
plot_k_distance_graph_full(df, k=10, eps_highlight=0.16)

# print("DBSCAN") 
# c1 = build_genotype_clusters(df=df, eps=0.16, min_samples=10)
# check_cluester_out(c1)
# diagnose_clusters(c1) 
    

# # Executa para os 5 maiores clusters do HDBSCAN
# inspect_top_clusters(c_hdb, top_n=15)

In [ ]:
viz.print_global_hamming_distance_per_seed(logdir="/home/flalipe/Projects/SoftRobots/log/EXP1-300gen/completeData.parquet")

In [ ]:
viz.print_hammig_inter_intra_task(logdir="/home/flalipe/Projects/SoftRobots/log/EXP1-300gen/completeData.parquet")